# Quantra Month 2: Quantum Portfolio Optimization via QAOA

In this notebook, we extend the classical Markowitz framework from Month 1 by formulating portfolio optimization as a **Quadratic Unconstrained Binary Optimization (QUBO)** problem. We then solve it using the **Quantum Approximate Optimization Algorithm (QAOA)** via Qiskit.

## 1. Problem Formulation
The discrete portfolio optimization problem seeks to select exactly $K$ assets out of $N$ to maximize returns while minimizing risk. We assign a binary variable $x_i \in \{0, 1\}$ to each asset.

**Objective function:**
$$\min_{x} \left[ \lambda \sum_{i,j} \sigma_{ij} x_i x_j - (1 - \lambda) \sum_i \mu_i x_i \right]$$

**Cardinality constraint:**
$$ \sum_i x_i = K $$

We enforce this constraint by adding a penalty term $P \left( \sum x_i - K \right)^2$, transforming it into an unconstrained QUBO.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.data_loader import fetch_nifty50_prices, get_returns
from src.quantum.problem_formulator import PortfolioQUBO
from src.quantum.qaoa_circuit import QAOACircuit
from src.quantum.qaoa_optimizer import QAOAOptimizer
from src.quantum.benchmarker import QuantumClassicalBenchmark

# Fetch subset of Nifty 50 for testing (e.g. 10 stocks)
tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS", 
           "WIPRO.NS", "AXISBANK.NS", "KOTAKBANK.NS", "LT.NS", "SBIN.NS"]
prices = fetch_nifty50_prices(tickers, "2022-01-01", "2024-01-01")
returns_df = get_returns(prices)
print(f"Loaded {len(tickers)} tickers.")

## 2. QUBO Matrix Construction
We instantiate the `PortfolioQUBO` class. Notice how the penalty term inflates diagonal elements to force standard states.

In [ ]:
K_assets = 5
qubo_maker = PortfolioQUBO(returns_df, n_assets_to_select=K_assets, risk_factor=0.5)
Q = qubo_maker.build_qubo_matrix()

plt.figure(figsize=(8,6))
plt.imshow(Q, cmap='coolwarm')
plt.colorbar(label='Coefficient')
plt.title('QUBO Matrix Heatmap')
plt.show()

## 3. QAOA Circuit Architecture
QAOA repeatedly applies a Problem Unitary $U_c(\gamma)$ and a Mixer Unitary $U_m(\beta)$ for $p$ layers.

In [ ]:
p_layers = 2
qaoa_circ = QAOACircuit(Q, p_layers=p_layers)
info = qaoa_circ.get_circuit_info()
print(f"Qubits: {info['n_qubits']}, Layers: {info['p_layers']}, Gates: {info['gate_count']}, Depth: {info['circuit_depth']}")
print("\nCircuit Diagram Snippet:\n")
print(info['circuit_diagram'])

## 4. Variational Optimization
We use COBYLA to optimize the $\gamma$ and $\beta$ parameters.

In [ ]:
optimizer = QAOAOptimizer(qaoa_circ, Q, n_shots=2048)
opt_results = optimizer.optimize(max_iterations=100)

print(f"Optimal Bitstring: {opt_results['optimal_bitstring']}")
print(f"Final Energy: {opt_results['final_energy']:.2f}")

# Decode
portfolio = qubo_maker.decode_bitstring(opt_results['optimal_bitstring'])
print(f"\nSelected Stocks: {portfolio['selected_stocks']}")
print(f"Expected Return: {portfolio['expected_return']*100:.2f}%")
print(f"Sharpe Ratio: {portfolio['sharpe_ratio']:.2f}")

## 5. Quantum vs Classical Benchmark
Using the full benchmarker pipeline to output rigorous comparative metrics.

In [ ]:
from src.utils.visualizer import set_plot_style, plot_quantum_vs_classical
from src.portfolio.markowitz import efficient_frontier

set_plot_style()
benchmarker = QuantumClassicalBenchmark(returns_df, tickers)
benchmark_res = benchmarker.compare(n_assets_to_select=K_assets, p_layers=p_layers, max_iterations=60)

print("\nImprovement Metrics:")
for k, v in benchmark_res['improvement'].items():
    print(f"{k}: {v}")

# Display 4-panel benchmark (assuming results dir exists or not saving)
frontier_df = efficient_frontier(benchmarker.mu, benchmarker.Sigma)
plot_quantum_vs_classical(benchmark_res, frontier_df, save_path=None)